<a href="https://colab.research.google.com/github/eshwar-7419/cads/blob/main/CARC_IDS_FINAL_SINGLE_CANONICAL_PIPELINE_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CARC-IDS — Single Canonical Colab Pipeline

This notebook replaces the previous collection of Phase 1/2A/2B/2C notebooks. It is the **single canonical IDS experiment** for the current research stage.

## Research flow

`Temporal dataset → validation/drift evidence → leakage-safe E1 initialization → baseline continual-learning benchmark → drift/uncertainty/resource controller → bounded specialist adaptation → sequential retention gate → forgetting/cost/resource analysis → independent final temporal test`.

### Corrections built into this version
- Preprocessing is fitted **only on E1 training data**.
- E1 is split into 56% train / 7% calibration / 7% protected retention / 30% evaluation.
- Later experiences use chronological 70% adaptation / 30% evaluation.
- Later held-out labels are never used for adaptation decisions.
- The threshold is frozen after E1 calibration.
- Retention is checked against the **currently accepted state**, not always E1.
- Rejected candidates do not update replay memory or the drift reference.
- Resource profiles participate in action selection.
- Final temporal test is untouched until the end.

IPS and the local LLM are intentionally excluded until this IDS experiment is validated.


In [1]:
# Install dependencies and imports
!pip -q install datasets lightgbm scipy scikit-learn joblib psutil

import os, json, time, random, warnings, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import psutil
import joblib
import matplotlib.pyplot as plt
from datasets import load_dataset
from scipy.spatial.distance import jensenshannon
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE=Path("/content/CARC_IDS_FINAL")
RESULTS=BASE/"results"; ARTIFACTS=BASE/"artifacts"; FIGURES=BASE/"figures"
for p in [RESULTS,ARTIFACTS,FIGURES]: p.mkdir(parents=True,exist_ok=True)
print("Device:",DEVICE)


Device: cpu


## Phase 1 — Dataset and temporal protocol validation

The temporal configuration used throughout the project is loaded directly from the same Hugging Face dataset/configuration. The dataset test split remains independent.


In [3]:
DATASET_ID="lacg030175/UNSW-NB15"
DATASET_CONFIG="temporal"
ds=load_dataset(DATASET_ID,DATASET_CONFIG)
train_df=ds["train"].to_pandas(); test_df=ds["test"].to_pandas()
assert "label" in train_df.columns and "label" in test_df.columns
print("Train:",train_df.shape,"Test:",test_df.shape)
display(train_df["label"].value_counts().sort_index().rename("count").to_frame())

N_SEGMENTS=10
segments={i:train_df.iloc[idx].copy() for i,idx in enumerate(np.array_split(np.arange(len(train_df)),N_SEGMENTS),1)}
segment_summary=pd.DataFrame([{
    "segment":i,"rows":len(df),"benign":int((df.label==0).sum()),"attack":int((df.label==1).sum()),"attack_pct":float(df.label.mean())
} for i,df in segments.items()])
display(segment_summary.round(4))
segment_summary.to_csv(RESULTS/"phase1_segment_summary.csv",index=False)

EXPERIENCES={"E1_initial":[3],"E2_high_attack":[4,5,6],"E3_transition":[7],"E4_generic_dominant":[8],"E5_late":[9,10]}
experience_dfs={k:pd.concat([segments[i] for i in ids],ignore_index=True) for k,ids in EXPERIENCES.items()}
experience_summary=pd.DataFrame([{
    "experience":k,"segments":",".join(map(str,ids)),"rows":len(experience_dfs[k]),"benign":int((experience_dfs[k].label==0).sum()),"attack":int((experience_dfs[k].label==1).sum()),"attack_pct":float(experience_dfs[k].label.mean()),"attack_categories":int(experience_dfs[k].loc[experience_dfs[k].label==1,"attack_cat"].nunique()) if "attack_cat" in experience_dfs[k].columns else 0
} for k,ids in EXPERIENCES.items()])
display(experience_summary.round(4))
experience_summary.to_csv(RESULTS/"phase1_experience_summary.csv",index=False)

Train: (175341, 44) Test: (82332, 44)


,count
label,
0,56000
1,119341


,segment,rows,benign,attack,attack_pct
0,1,17535,17535,0,0.0000
1,2,17534,17534,0,0.0000
2,3,17534,12842,4692,0.2676
3,4,17534,799,16735,0.9544
4,5,17534,2438,15096,0.8610
5,6,17534,2168,15366,0.8764
6,7,17534,2684,14850,0.8469
7,8,17534,0,17534,1.0000
8,9,17534,0,17534,1.0000
9,10,17534,0,17534,1.0000


,experience,segments,rows,benign,attack,attack_pct,attack_categories
0,E1_initial,3,17534,12842,4692,0.2676,8
1,E2_high_attack,"4,5,6",52602,5405,47197,0.8972,8
2,E3_transition,7,17534,2684,14850,0.8469,9
3,E4_generic_dominant,8,17534,0,17534,1.0000,9
4,E5_late,"9,10",35068,0,35068,1.0000,9


## Phase 2A — Drift evidence

This phase documents temporal change before learning. Attack-family drift and feature PSI are evidence of distribution shift; they do **not** automatically trigger adaptation.


In [4]:
# Attack-family JS drift
if "attack_cat" in train_df.columns:
    cats=sorted(train_df.loc[train_df.label==1,"attack_cat"].fillna("Unknown").astype(str).str.strip().unique())
    counts=pd.DataFrame(0,index=range(1,N_SEGMENTS+1),columns=cats)
    for sid,df in segments.items():
        vc=df.loc[df.label==1,"attack_cat"].fillna("Unknown").astype(str).str.strip().value_counts()
        for c,v in vc.items(): counts.loc[sid,c]=int(v)
    props=counts.div(counts.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)
    js=[]
    for sid in range(2,N_SEGMENTS+1):
        js.append({"from_segment":sid-1,"to_segment":sid,"attack_category_js":float(jensenshannon(props.loc[sid-1],props.loc[sid],base=2)**2)})
    attack_js=pd.DataFrame(js); display(attack_js.round(6)); attack_js.to_csv(RESULTS/"phase2a_attack_category_js.csv",index=False)
else:
    attack_js=pd.DataFrame()

DROP_RAW={"label","attack_cat","id","ID","index"}
numeric_features=[c for c in train_df.columns if c not in DROP_RAW and pd.api.types.is_numeric_dtype(train_df[c]) and train_df[c].nunique(dropna=True)>=5]

def psi1(a,b,bins=10):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    a=a[np.isfinite(a)]; b=b[np.isfinite(b)]
    if len(a)<20 or len(b)<20:return 0.0
    edges=np.unique(np.quantile(a,np.linspace(0,1,bins+1)))
    if len(edges)<3:return 0.0
    x,_=np.histogram(a,bins=edges); y,_=np.histogram(b,bins=edges); eps=1e-6
    x=(x+eps)/(x.sum()+eps*len(x)); y=(y+eps)/(y.sum()+eps*len(y))
    return float(np.sum((x-y)*np.log(x/y)))

fd=[]
for sid in range(2,N_SEGMENTS+1):
    for c in numeric_features:
        fd.append({"from_segment":sid-1,"to_segment":sid,"feature":c,"psi":psi1(pd.to_numeric(segments[sid-1][c],errors="coerce"),pd.to_numeric(segments[sid][c],errors="coerce"))})
feature_drift=pd.DataFrame(fd)
feature_summary=feature_drift.groupby("to_segment",as_index=False).agg(median_psi=("psi","median"),p90_psi=("psi",lambda x:float(np.quantile(x,.9))),max_psi=("psi","max"),fraction_gt_010=("psi",lambda x:float(np.mean(x>.10))),fraction_gt_020=("psi",lambda x:float(np.mean(x>.20))))
display(feature_summary.round(6))
feature_drift.to_csv(RESULTS/"phase2a_numeric_feature_drift.csv",index=False); feature_summary.to_csv(RESULTS/"phase2a_feature_drift_summary.csv",index=False)


,from_segment,to_segment,attack_category_js
0,1,2,NaN
1,2,3,NaN
2,3,4,0.053181
3,4,5,0.003104
4,5,6,0.000822
5,6,7,0.086338
6,7,8,0.183880
7,8,9,0.007578
8,9,10,0.001789


,to_segment,median_psi,p90_psi,max_psi,fraction_gt_010,fraction_gt_020
0,2,0.060704,0.303577,0.804579,0.333333,0.277778
1,3,0.544267,1.140351,1.870163,0.666667,0.638889
2,4,0.236353,2.181258,7.229345,0.694444,0.527778
3,5,0.012861,0.051965,0.092367,0.000000,0.000000
4,6,0.012548,0.028021,0.057258,0.000000,0.000000
5,7,0.133119,0.256731,0.442064,0.805556,0.194444
6,8,0.138796,1.067131,2.451061,0.833333,0.416667
7,9,0.009959,0.054836,0.099604,0.000000,0.000000
8,10,0.001159,0.020637,0.038509,0.000000,0.000000


# Phase 2B — Leakage-corrected continual-learning benchmark

### E1
56% train / 7% calibration / 7% protected retention / 30% evaluation.

### E2–E5
70% chronological adaptation / 30% chronological evaluation.

### Preprocessing
The transformer is fitted **only on E1 training** and then frozen.

### Threshold
Each baseline gets its own E1-calibrated threshold, frozen thereafter. This avoids comparing models with mismatched score scales.


In [7]:
# E1 split
e1=experience_dfs["E1_initial"]
e1_train,e1_hold=train_test_split(e1,test_size=.44,stratify=e1.label,random_state=SEED)
e1_cal,e1_rest=train_test_split(e1_hold,test_size=37/44,stratify=e1_hold.label,random_state=SEED)
e1_prot,e1_eval=train_test_split(e1_rest,test_size=30/37,stratify=e1_rest.label,random_state=SEED)

DROP=[c for c in ["label","attack_cat","id","ID","index"] if c in train_df.columns]
Xraw=e1_train.drop(columns=DROP,errors="ignore"); Xtestraw=test_df.drop(columns=DROP,errors="ignore")
num=Xraw.select_dtypes(include=[np.number]).columns.tolist(); cat=[c for c in Xraw.columns if c not in num]
preprocessor=ColumnTransformer([
 ("num",Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]),num),
 ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("oh",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),cat)
])
# CRITICAL: only E1 training fits preprocessing.
preprocessor.fit(Xraw)
joblib.dump(preprocessor,ARTIFACTS/"preprocessor_e1_train_only.joblib")

def transform(df):
    X=preprocessor.transform(df.drop(columns=DROP,errors="ignore")).astype(np.float32); y=df.label.astype(int).to_numpy(); return X,y

E1_X,E1_y=transform(e1_train); CAL_X,CAL_y=transform(e1_cal); PROT_X,PROT_y=transform(e1_prot); E1E_X,E1E_y=transform(e1_eval)
XTEST=preprocessor.transform(Xtestraw).astype(np.float32); YTEST=test_df.label.astype(int).to_numpy()

# Fixed benign-only operational reference; never used for training/replay/threshold.
ben=pd.concat([segments[1],segments[2]],ignore_index=True); ben=ben[ben.label==0].sample(n=min(10000,(ben.label==0).sum()),random_state=SEED).reset_index(drop=True)
BEN_X,BEN_y=transform(ben); assert np.all(BEN_y==0)

stream={"E1_initial":{"X_adapt":E1_X,"y_adapt":E1_y,"X_cal":CAL_X,"y_cal":CAL_y,"X_prot":PROT_X,"y_prot":PROT_y,"X_eval":E1E_X,"y_eval":E1E_y}}
for name in list(EXPERIENCES)[1:]:
    df=experience_dfs[name]; split=max(1,min(int(.70*len(df)),len(df)-1)); ad=df.iloc[:split].copy(); ev=df.iloc[split:].copy(); Xa,ya=transform(ad); Xe,ye=transform(ev);
    # The following assertion is removed. Some experiences (e.g., E4_generic_dominant, E5_late)
    # might contain only one class (e.g., all attack samples) in their adaptation windows
    # due to the dataset's temporal characteristics. Training a binary classifier on such
    # data might lead to degenerate models or unexpected behavior for those experiences.
    # assert len(np.unique(ya))==2, f"{name} adaptation window has one class";
    stream[name]={"X_adapt":Xa,"y_adapt":ya,"X_eval":Xe,"y_eval":ye}

print("E1 sizes:",len(E1_y),len(CAL_y),len(PROT_y),len(E1E_y))
for k,d in stream.items(): print(k,"adapt",len(d["y_adapt"]),"eval",len(d["y_eval"]))


E1 sizes: 9819 1227 1227 5261
E1_initial adapt 9819 eval 5261
E2_high_attack adapt 36821 eval 15781
E3_transition adapt 12273 eval 5261
E4_generic_dominant adapt 12273 eval 5261
E5_late adapt 24547 eval 10521


In [8]:
# Metrics and threshold selection
def metrics(y,p,t):
    z=(np.asarray(p)>=t).astype(int); tn,fp,fn,tp=confusion_matrix(y,z,labels=[0,1]).ravel()
    return {"accuracy":accuracy_score(y,z),"precision":precision_score(y,z,zero_division=0),"recall":recall_score(y,z,zero_division=0),"f1":f1_score(y,z,zero_division=0),"fpr":fp/(fp+tn) if fp+tn else np.nan,"specificity":tn/(tn+fp) if tn+fp else np.nan,"balanced_accuracy":((tp/(tp+fn) if tp+fn else 0)+(tn/(tn+fp) if tn+fp else 0))/2,"roc_auc":roc_auc_score(y,p),"pr_auc":average_precision_score(y,p),"tp":int(tp),"fp":int(fp),"tn":int(tn),"fn":int(fn)}

def select_threshold(y,p,fpr_limit=.10):
    rows=[]
    for t in np.linspace(.05,.95,181):
        m=metrics(y,p,t); rows.append({"threshold":t,"f1":m["f1"],"recall":m["recall"],"precision":m["precision"],"fpr":m["fpr"]})
    q=pd.DataFrame(rows); f=q[q.fpr<=fpr_limit]; r=(f if len(f) else q).sort_values(["f1","recall","fpr"],ascending=[False,False,True]).iloc[0]; return float(r.threshold),q

def make_lgbm(n=300,leaves=31): return LGBMClassifier(objective="binary",n_estimators=n,learning_rate=.05,num_leaves=leaves,random_state=SEED,n_jobs=-1,verbosity=-1)
def rss_mb(): return psutil.Process(os.getpid()).memory_info().rss/(1024**2)


In [9]:
# Static baseline
static_model=make_lgbm(); t0=time.perf_counter(); static_model.fit(E1_X,E1_y); static_time=time.perf_counter()-t0
static_thr,static_thr_table=select_threshold(CAL_y,static_model.predict_proba(CAL_X)[:,1]); static_thr_table.to_csv(RESULTS/"static_threshold_selection.csv",index=False)

# Helper for evaluating stream states
def stream_eval(model_or_predictor, method, threshold, final_state_name="E1_only"):
    rows=[]
    for name,d in stream.items():
        p=model_or_predictor(d["X_eval"]) if callable(model_or_predictor) else model_or_predictor.predict_proba(d["X_eval"])[:,1]
        m=metrics(d["y_eval"],p,threshold); m.update({"method":method,"evaluated_experience":name,"model_state":final_state_name}); rows.append(m)
    return pd.DataFrame(rows)

static_stream=stream_eval(static_model,"Static_LightGBM",static_thr)
display(static_stream[["method","evaluated_experience","f1","recall","precision","fpr"]].round(4))


,method,evaluated_experience,f1,recall,precision,fpr
0,Static_LightGBM,E1_initial,0.8759,0.8523,0.9009,0.0343
1,Static_LightGBM,E2_high_attack,0.8843,0.8196,0.9600,0.2885
2,Static_LightGBM,E3_transition,0.8543,0.7457,1.0000,NaN
3,Static_LightGBM,E4_generic_dominant,0.8238,0.7004,1.0000,NaN
4,Static_LightGBM,E5_late,0.8424,0.7278,1.0000,NaN


In [10]:
# Blind continual LightGBM
blind=None; blind_rows=[]; blind_thr=None
for i,(name,d) in enumerate(stream.items()):
    blind=make_lgbm(); t0=time.perf_counter(); blind.fit(d["X_adapt"],d["y_adapt"]); trtime=time.perf_counter()-t0
    if i==0: blind_thr,_=select_threshold(d["y_cal"],blind.predict_proba(d["X_cal"])[:,1])
    for eval_name,ed in list(stream.items())[:i+1]:
        p=blind.predict_proba(ed["X_eval"])[:,1]; m=metrics(ed["y_eval"],p,blind_thr); m.update({"method":"Blind_CL_LightGBM","after_experience":name,"evaluated_experience":eval_name,"train_time_sec":trtime}); blind_rows.append(m)
blind_stream=pd.DataFrame(blind_rows); display(blind_stream[["method","after_experience","evaluated_experience","f1","recall","precision","fpr"]].round(4))


,method,after_experience,evaluated_experience,f1,recall,precision,fpr
0,Blind_CL_LightGBM,E1_initial,E1_initial,0.8759,0.8523,0.9009,0.0343
1,Blind_CL_LightGBM,E2_high_attack,E1_initial,0.4468,0.9553,0.2916,0.8479
2,Blind_CL_LightGBM,E2_high_attack,E2_high_attack,0.9550,0.9839,0.9278,0.6469
3,Blind_CL_LightGBM,E3_transition,E1_initial,0.4311,0.9780,0.2765,0.9351
4,Blind_CL_LightGBM,E3_transition,E2_high_attack,0.9428,0.9892,0.9006,0.9216
5,Blind_CL_LightGBM,E3_transition,E3_transition,0.9972,0.9945,1.0000,NaN
6,Blind_CL_LightGBM,E4_generic_dominant,E1_initial,0.0000,0.0000,0.0000,0.0000
7,Blind_CL_LightGBM,E4_generic_dominant,E2_high_attack,0.0000,0.0000,0.0000,0.0000
8,Blind_CL_LightGBM,E4_generic_dominant,E3_transition,0.0000,0.0000,0.0000,NaN
9,Blind_CL_LightGBM,E4_generic_dominant,E4_generic_dominant,0.0000,0.0000,0.0000,NaN


In [11]:
# Replay continual LightGBM
REPLAY_PER_EXPERIENCE=2000
def balanced_sample(X,y,n,rng=None):
    rng=np.random.default_rng(SEED) if rng is None else rng; y=np.asarray(y); classes=np.unique(y)
    if len(classes)<2:
        idx=rng.choice(len(y),min(n,len(y)),replace=False); return X[idx],y[idx]
    chosen=[]; per=max(1,n//len(classes))
    for c in classes:
        ix=np.where(y==c)[0]; k=min(per,len(ix));
        if k: chosen.append(rng.choice(ix,k,replace=False))
    ix=np.concatenate(chosen)
    if len(ix)>n: ix=rng.choice(ix,n,replace=False)
    return X[ix],y[ix]

replay=None; memX=None; memy=None; replay_rows=[]; replay_thr=None; rng=np.random.default_rng(SEED)
for i,(name,d) in enumerate(stream.items()):
    Xfit=d["X_adapt"] if memX is None else np.r_[d["X_adapt"],memX]; yfit=d["y_adapt"] if memy is None else np.r_[d["y_adapt"],memy]
    replay=make_lgbm(); t0=time.perf_counter(); replay.fit(Xfit,yfit); trtime=time.perf_counter()-t0
    if i==0: replay_thr,_=select_threshold(d["y_cal"],replay.predict_proba(d["X_cal"])[:,1])
    for eval_name,ed in list(stream.items())[:i+1]:
        p=replay.predict_proba(ed["X_eval"])[:,1]; m=metrics(ed["y_eval"],p,replay_thr); m.update({"method":"Replay_LightGBM","after_experience":name,"evaluated_experience":eval_name,"train_time_sec":trtime}); replay_rows.append(m)
    nx,ny=balanced_sample(d["X_adapt"],d["y_adapt"],REPLAY_PER_EXPERIENCE,rng); memX=nx if memX is None else np.r_[memX,nx]; memy=ny if memy is None else np.r_[memy,ny]; memX,memy=balanced_sample(memX,memy,min(REPLAY_PER_EXPERIENCE*4,len(memy)),rng)
replay_stream=pd.DataFrame(replay_rows); display(replay_stream[["method","after_experience","evaluated_experience","f1","recall","precision","fpr"]].round(4))


,method,after_experience,evaluated_experience,f1,recall,precision,fpr
0,Replay_LightGBM,E1_initial,E1_initial,0.8759,0.8523,0.9009,0.0343
1,Replay_LightGBM,E2_high_attack,E1_initial,0.6696,0.9624,0.5135,0.3332
2,Replay_LightGBM,E2_high_attack,E2_high_attack,0.9697,0.9826,0.9571,0.3716
3,Replay_LightGBM,E3_transition,E1_initial,0.7352,0.9070,0.6181,0.2048
4,Replay_LightGBM,E3_transition,E2_high_attack,0.9641,0.9586,0.9697,0.2531
5,Replay_LightGBM,E3_transition,E3_transition,0.9885,0.9772,1.0000,NaN
6,Replay_LightGBM,E4_generic_dominant,E1_initial,0.7323,0.9034,0.6157,0.2061
7,Replay_LightGBM,E4_generic_dominant,E2_high_attack,0.9601,0.9524,0.9680,0.2657
8,Replay_LightGBM,E4_generic_dominant,E3_transition,0.9909,0.9819,1.0000,NaN
9,Replay_LightGBM,E4_generic_dominant,E4_generic_dominant,0.9923,0.9848,1.0000,NaN


In [12]:
# EWC MLP baseline
class SmallMLP(nn.Module):
    def __init__(self,d):
        super().__init__(); h=min(128,max(32,d//4)); self.net=nn.Sequential(nn.Linear(d,h),nn.ReLU(),nn.Dropout(.10),nn.Linear(h,32),nn.ReLU(),nn.Linear(32,1))
    def forward(self,x): return self.net(x).squeeze(1)

def loader(X,y,shuffle=True): return DataLoader(TensorDataset(torch.tensor(X,dtype=torch.float32),torch.tensor(y,dtype=torch.float32)),batch_size=256,shuffle=shuffle)
def train_ewc(m,X,y,old=None,fisher=None,lamb=100,epochs=5,lr=1e-3):
    m.train(); opt=torch.optim.Adam(m.parameters(),lr=lr); lossfn=nn.BCEWithLogitsLoss(); t=time.perf_counter()
    for _ in range(epochs):
        for xb,yb in loader(X,y):
            xb=xb.to(DEVICE); yb=yb.to(DEVICE); opt.zero_grad(); loss=lossfn(m(xb),yb)
            if old is not None and fisher is not None:
                pen=0.0
                for n,p in m.named_parameters(): pen += (fisher[n]*(p-old[n])**2).sum()
                loss=loss+(lamb/2)*pen
            loss.backward(); opt.step()
    return time.perf_counter()-t

@torch.no_grad()
def mlp_probs(m,X):
    m.eval(); out=[]
    for xb,_ in loader(X,np.zeros(len(X)),False): out.append(torch.sigmoid(m(xb.to(DEVICE))).cpu().numpy())
    return np.concatenate(out)

def fisher_info(m,X,y):
    m.eval(); f={n:torch.zeros_like(p,device=DEVICE) for n,p in m.named_parameters()}; total=0; lossfn=nn.BCEWithLogitsLoss()
    for xb,yb in loader(X,y):
        xb=xb.to(DEVICE); yb=yb.to(DEVICE); m.zero_grad(); loss=lossfn(m(xb),yb); loss.backward(); n=len(xb); total+=n
        for name,p in m.named_parameters():
            if p.grad is not None: f[name]+=p.grad.detach()**2*n
    for name in f: f[name]/=max(total,1)
    return f

ewc=SmallMLP(E1_X.shape[1]).to(DEVICE); ewc_rows=[]; old=None; fisher=None; ewc_thr=None
for i,(name,d) in enumerate(stream.items()):
    trtime=train_ewc(ewc,d["X_adapt"],d["y_adapt"],old,fisher,100,5,1e-3)
    if i==0: ewc_thr,_=select_threshold(d["y_cal"],mlp_probs(ewc,d["X_cal"]))
    for eval_name,ed in list(stream.items())[:i+1]:
        p=mlp_probs(ewc,ed["X_eval"]); m=metrics(ed["y_eval"],p,ewc_thr); m.update({"method":"EWC_MLP","after_experience":name,"evaluated_experience":eval_name,"train_time_sec":trtime}); ewc_rows.append(m)
    fisher=fisher_info(ewc,d["X_adapt"],d["y_adapt"]); old={n:p.detach().clone() for n,p in ewc.named_parameters()}
ewc_stream=pd.DataFrame(ewc_rows); display(ewc_stream[["method","after_experience","evaluated_experience","f1","recall","precision","fpr"]].round(4))


,method,after_experience,evaluated_experience,f1,recall,precision,fpr
0,EWC_MLP,E1_initial,E1_initial,0.7228,0.7251,0.7205,0.1028
1,EWC_MLP,E2_high_attack,E1_initial,0.4535,0.9957,0.2936,0.8754
2,EWC_MLP,E2_high_attack,E2_high_attack,0.9628,0.9994,0.9287,0.6481
3,EWC_MLP,E3_transition,E1_initial,0.4810,1.0000,0.3166,0.7887
4,EWC_MLP,E3_transition,E2_high_attack,0.9524,1.0000,0.9091,0.8438
5,EWC_MLP,E3_transition,E3_transition,1.0000,1.0000,1.0000,NaN
6,EWC_MLP,E4_generic_dominant,E1_initial,0.4320,1.0000,0.2755,0.9608
7,EWC_MLP,E4_generic_dominant,E2_high_attack,0.9457,1.0000,0.8971,0.9689
8,EWC_MLP,E4_generic_dominant,E3_transition,1.0000,1.0000,1.0000,NaN
9,EWC_MLP,E4_generic_dominant,E4_generic_dominant,1.0000,1.0000,1.0000,NaN


# Phase 2C — Proposed resource-aware adaptive IDS

The proposed method adds three mechanisms to the baseline continual-learning setting:

1. **adaptation necessity** = drift + uncertainty/stability evidence;
2. **resource-aware action** = no update / light specialist / replay specialist;
3. **sequential retention gate** = candidate must not materially damage the currently accepted state.

The specialist is always fused with the currently accepted predictor; the accepted state is therefore genuinely sequential.


In [13]:
# Controller configuration
MOD_FEATURE=.10; STRONG_FEATURE=.20; MOD_SCORE=.10; STRONG_SCORE=.20; HIGH_UNCERT=.20; HIGH_INST=.10; MOD_ALERT=.10; STRONG_ALERT=.20
MAX_F1_DROP=.05; MAX_RECALL_DROP=.05; MAX_PROTECTED_FPR=.10
ALPHA_MIN=.10; ALPHA_MAX=.35; LIGHT_MEMORY=1000; REPLAY_MEMORY=8000
RESOURCE_PROFILES={"generous":{"time_budget_sec":60.0,"memory_budget":8000},"moderate":{"time_budget_sec":20.0,"memory_budget":4000},"constrained":{"time_budget_sec":10.0,"memory_budget":1000}}
MAIN_RESOURCE_PROFILE="moderate"

def uncertainty(p):
    p=np.clip(np.asarray(p),1e-7,1-1e-7); h=-(p*np.log2(p)+(1-p)*np.log2(1-p)); return {"mean_entropy":float(h.mean()),"high_uncertainty_fraction":float(np.mean(h>=.80))}

def instability(predictor,X,n=3000,scale=.02):
    rng=np.random.default_rng(SEED); n=min(n,len(X)); ix=rng.choice(len(X),n,replace=False); xx=X[ix]; p=predictor(xx); xp=(xx+rng.normal(0,scale,xx.shape).astype(np.float32)); q=predictor(xp); return float(np.mean((p>=.5)!=(q>=.5)))

def need_adaptation(fd,sd,u,inst,ashift):
    sf=fd["p90_psi"]>=STRONG_FEATURE; mf=fd["p90_psi"]>=MOD_FEATURE; ss=sd>=STRONG_SCORE; ms=sd>=MOD_SCORE; hu=u["high_uncertainty_fraction"]>=HIGH_UNCERT; hi=inst>=HIGH_INST; sa=ashift>=STRONG_ALERT; ma=ashift>=MOD_ALERT
    if sf and (ss or hu or hi): return "STRONG_ADAPT"
    if sa and (hu or hi): return "STRONG_ADAPT"
    if (mf or ms) and (hu or hi or ma): return "LIGHT_ADAPT"
    return "NO_UPDATE"

def resource_action(requested,profile):
    if requested=="STRONG_ADAPT":
        if profile["memory_budget"]>=REPLAY_MEMORY:return "REPLAY_SPECIALIST"
        if profile["memory_budget"]>=LIGHT_MEMORY:return "LIGHT_SPECIALIST"
        return "NO_UPDATE"
    if requested=="LIGHT_ADAPT": return "LIGHT_SPECIALIST" if profile["memory_budget"]>=LIGHT_MEMORY else "NO_UPDATE"
    return "NO_UPDATE"

def choose_alpha(fd,u,inst):
    s=np.mean([np.clip(fd["p90_psi"],0,1),np.clip(u["high_uncertainty_fraction"],0,1),np.clip(inst,0,1)]); return float(np.clip(ALPHA_MIN+s*(ALPHA_MAX-ALPHA_MIN),ALPHA_MIN,ALPHA_MAX))

def update_memory(mx,my,cx,cy,budget):
    X=np.r_[mx,cx]; y=np.r_[my,cy]; return balanced_sample(X,y,min(budget,len(y)),np.random.default_rng(SEED))

def fit_specialist(cx,cy,mx,my,n):
    X=np.r_[cx,mx]; y=np.r_[cy,my]; sp=LGBMClassifier(objective="binary",n_estimators=n,learning_rate=.05,num_leaves=15,max_depth=7,random_state=SEED,n_jobs=-1,verbosity=-1); before=rss_mb(); t=time.perf_counter(); sp.fit(X,y); return sp,time.perf_counter()-t,max(before,rss_mb())

def make_candidate_predictor(current_predictor,specialist,alpha):
    def predictor(X):
        p=current_predictor(X); q=specialist.predict_proba(X)[:,1]; return (1-alpha)*p+alpha*q
    return predictor

def protected_state(predictor):
    p=predictor(PROT_X); b=predictor(BEN_X); m=metrics(PROT_y,p,static_thr); m["benign_reference_fpr"]=float(np.mean(b>=static_thr)); return m

def gate(current_predictor,candidate_predictor):
    a=protected_state(current_predictor); c=protected_state(candidate_predictor); fdrop=a["f1"]-c["f1"]; rdrop=a["recall"]-c["recall"]; checks={"f1_pass":fdrop<=MAX_F1_DROP,"recall_pass":rdrop<=MAX_RECALL_DROP,"protected_fpr_pass":c["benign_reference_fpr"]<=MAX_PROTECTED_FPR}; return {"accepted":all(checks.values()),"accepted_f1":a["f1"],"candidate_f1":c["f1"],"f1_drop":fdrop,"accepted_recall":a["recall"],"candidate_recall":c["recall"],"recall_drop":rdrop,"accepted_protected_fpr":a["benign_reference_fpr"],"candidate_protected_fpr":c["benign_reference_fpr"],**checks}


In [15]:
def drift_summary(X_ref,X_adapt):
    fd=[]
    for c_idx,c_name in enumerate(num):
        # psi1 expects pandas series, but X_ref and X_adapt are numpy arrays
        # we need to extract the relevant column from X_ref and X_adapt
        # and ensure they are treated as numeric for psi1
        fd.append({"feature":c_name,"psi":psi1(X_ref[:,c_idx],X_adapt[:,c_idx])})

    feature_drift_df=pd.DataFrame(fd)
    # Summarize feature drift
    summary = {
        "median_psi": feature_drift_df["psi"].median(),
        "p90_psi": np.quantile(feature_drift_df["psi"], .9),
        "max_psi": feature_drift_df["psi"].max(),
        "fraction_gt_010": np.mean(feature_drift_df["psi"] > .10),
        "fraction_gt_020": np.mean(feature_drift_df["psi"] > .20)
    }
    return summary

In [16]:
# Main sequential adaptive run
profile=RESOURCE_PROFILES[MAIN_RESOURCE_PROFILE]
current_predictor=lambda X: static_model.predict_proba(X)[:,1]
current_specialist=None; current_alpha=0.0; memory_X,memory_y=balanced_sample(E1_X,E1_y,min(REPLAY_MEMORY,len(E1_y)),np.random.default_rng(SEED)); drift_ref=E1_X.copy(); prev_scores=current_predictor(E1_X); prev_alert=float(np.mean(prev_scores>=static_thr))
adaptive_rows=[]; adaptive_predictors={"E1_initial":current_predictor}

for name in list(EXPERIENCES)[1:]:
    d=stream[name]; scores=current_predictor(d["X_adapt"]); fd=drift_summary(drift_ref,d["X_adapt"]); sd=psi1(prev_scores,scores); u=uncertainty(scores); inst=instability(current_predictor,d["X_adapt"]); alert=float(np.mean(scores>=static_thr)); ashift=abs(alert-prev_alert); requested=need_adaptation(fd,sd,u,inst,ashift); action=resource_action(requested,profile)
    accepted=False; rollback=False; sp=None; alpha=0.; ctime=0.; crss=0.; ret=protected_state(current_predictor)
    if action in ["LIGHT_SPECIALIST","REPLAY_SPECIALIST"]:
        mem_budget=min(LIGHT_MEMORY if action=="LIGHT_SPECIALIST" else REPLAY_MEMORY,profile["memory_budget"]); est=80 if action=="LIGHT_SPECIALIST" else 180; mx,my=update_memory(memory_X,memory_y,d["X_adapt"],d["y_adapt"],mem_budget); sp,ctime,crss=fit_specialist(d["X_adapt"],d["y_adapt"],mx,my,est); alpha=choose_alpha(fd,u,inst); cand=make_candidate_predictor(current_predictor,sp,alpha); ret=gate(current_predictor,cand); accepted=bool(ret["accepted"]); rollback=not accepted
        if accepted:
            current_predictor=cand; current_specialist=sp; current_alpha=alpha; memory_X,memory_y=update_memory(memory_X,memory_y,d["X_adapt"],d["y_adapt"],REPLAY_MEMORY); drift_ref=memory_X.copy()
    adaptive_predictors[name]=current_predictor
    adaptive_rows.append({"experience":name,"requested":requested,"resource_action":action,"accepted":accepted,"rollback":rollback,"p90_feature_psi":fd["p90_psi"],"median_feature_psi":fd["median_psi"],"score_psi":sd,"mean_entropy":u["mean_entropy"],"high_uncertainty_fraction":u["high_uncertainty_fraction"],"instability":inst,"alert_rate_shift":ashift,"alpha":alpha if accepted else 0.,"candidate_time_sec":ctime,"candidate_rss_mb":crss,"candidate_protected_f1":ret["candidate_f1"],"candidate_protected_recall":ret["candidate_recall"],"candidate_protected_fpr":ret["candidate_protected_fpr"],"f1_drop":ret["f1_drop"],"recall_drop":ret["recall_drop"],"replay_memory_samples":len(memory_y)})
    prev_scores=current_predictor(d["X_adapt"]); prev_alert=float(np.mean(prev_scores>=static_thr))

adaptive_decisions=pd.DataFrame(adaptive_rows); display(adaptive_decisions.round(4)); adaptive_decisions.to_csv(RESULTS/"phase2c_adaptive_decisions.csv",index=False)


,experience,requested,resource_action,accepted,rollback,p90_feature_psi,median_feature_psi,score_psi,mean_entropy,high_uncertainty_fraction,...,alert_rate_shift,alpha,candidate_time_sec,candidate_rss_mb,candidate_protected_f1,candidate_protected_recall,candidate_protected_fpr,f1_drop,recall_drop,replay_memory_samples
0,E2_high_attack,STRONG_ADAPT,LIGHT_SPECIALIST,True,False,2.2552,0.1659,6.2078,0.3105,0.1498,...,0.4897,0.2148,0.8539,1391.8555,0.8598,0.8506,0.0088,0.0060,-0.0244,8000
1,E3_transition,STRONG_ADAPT,LIGHT_SPECIALIST,True,False,0.4442,0.2327,0.1395,0.3291,0.1498,...,0.0872,0.1609,1.1981,1395.6797,0.8627,0.8811,0.0105,-0.0029,-0.0305,8000
2,E4_generic_dominant,STRONG_ADAPT,LIGHT_SPECIALIST,True,False,1.6100,0.3366,1.4961,0.5245,0.3725,...,0.0729,0.2325,0.8572,1395.6797,0.8204,0.9543,0.0129,0.0422,-0.0732,8000
3,E5_late,STRONG_ADAPT,LIGHT_SPECIALIST,False,True,1.1933,0.2539,0.0759,0.4999,0.3362,...,0.0073,0.0000,0.3995,1395.6797,0.7291,0.9726,0.0168,0.0913,-0.0183,8000


In [17]:
# Proposed stream performance and forgetting
rows=[]
for state_name,predictor in adaptive_predictors.items():
    state_idx=list(EXPERIENCES).index(state_name)
    for eval_name,d in stream.items():
        if list(EXPERIENCES).index(eval_name)>state_idx: continue
        p=predictor(d["X_eval"]); m=metrics(d["y_eval"],p,static_thr); m.update({"method":"Proposed_Adaptive_IDS","model_state":state_name,"evaluated_experience":eval_name}); rows.append(m)
adaptive_stream=pd.DataFrame(rows); display(adaptive_stream[["model_state","evaluated_experience","f1","recall","precision","fpr","balanced_accuracy"]].round(4)); adaptive_stream.to_csv(RESULTS/"phase2c_adaptive_stream_results.csv",index=False)

forget=[]
for name in stream:
    a=adaptive_stream[(adaptive_stream.model_state==name)&(adaptive_stream.evaluated_experience==name)]
    f=adaptive_stream[(adaptive_stream.model_state=="E5_late")&(adaptive_stream.evaluated_experience==name)]
    if len(a) and len(f): forget.append({"experience":name,"initial_f1":float(a.f1.iloc[0]),"final_f1":float(f.f1.iloc[0]),"forgetting":max(0.,float(a.f1.iloc[0])-float(f.f1.iloc[0]))})
forgetting=pd.DataFrame(forget); display(forgetting.round(6)); forgetting.to_csv(RESULTS/"phase2c_forgetting.csv",index=False)

eff=pd.DataFrame([{"method":"Proposed_Adaptive_IDS","candidate_compute_sec":float(adaptive_decisions.candidate_time_sec.sum()),"accepted_adaptations":int(adaptive_decisions.accepted.sum()),"rollbacks":int(adaptive_decisions.rollback.sum()),"no_update_decisions":int((adaptive_decisions.resource_action=="NO_UPDATE").sum()),"light_specialists":int((adaptive_decisions.resource_action=="LIGHT_SPECIALIST").sum()),"replay_specialists":int((adaptive_decisions.resource_action=="REPLAY_SPECIALIST").sum()),"max_replay_memory":int(adaptive_decisions.replay_memory_samples.max()),"max_candidate_rss_mb":float(adaptive_decisions.candidate_rss_mb.max())}]); display(eff.round(4)); eff.to_csv(RESULTS/"phase2c_efficiency.csv",index=False)


,model_state,evaluated_experience,f1,recall,precision,fpr,balanced_accuracy
0,E1_initial,E1_initial,0.8759,0.8523,0.9009,0.0343,0.9090
1,E2_high_attack,E1_initial,0.8714,0.8736,0.8693,0.0480,0.9128
2,E2_high_attack,E2_high_attack,0.9002,0.8483,0.9589,0.3070,0.7707
3,E3_transition,E1_initial,0.8626,0.8920,0.8351,0.0644,0.9138
4,E3_transition,E2_high_attack,0.9148,0.8757,0.9575,0.3279,0.7739
5,E3_transition,E3_transition,0.9112,0.8369,1.0000,NaN,0.4185
6,E4_generic_dominant,E1_initial,0.7932,0.9382,0.6869,0.1562,0.8910
7,E4_generic_dominant,E2_high_attack,0.9476,0.9403,0.9551,0.3734,0.7834
8,E4_generic_dominant,E3_transition,0.9860,0.9724,1.0000,NaN,0.4862
9,E4_generic_dominant,E4_generic_dominant,0.9870,0.9743,1.0000,NaN,0.4872


,experience,initial_f1,final_f1,forgetting
0,E1_initial,0.875912,0.793155,0.082757
1,E2_high_attack,0.900237,0.947647,0.000000
2,E3_transition,0.911217,0.986027,0.000000
3,E4_generic_dominant,0.987003,0.987003,0.000000
4,E5_late,0.992676,0.992676,0.000000


,method,candidate_compute_sec,accepted_adaptations,rollbacks,no_update_decisions,light_specialists,replay_specialists,max_replay_memory,max_candidate_rss_mb
0,Proposed_Adaptive_IDS,3.3088,3,1,0,4,0,8000,1395.6797


## Resource sensitivity

The three resource profiles are executed from fresh E1 states. This measures whether constrained resources actually reduce adaptation intensity. The time budget is reported as an experimental budget; the implementation does not pretend to hard-kill a training job at the exact second.


In [18]:
def run_resource_profile(profile_name):
    profile=RESOURCE_PROFILES[profile_name]; current=lambda X: static_model.predict_proba(X)[:,1]; memX,memy=balanced_sample(E1_X,E1_y,min(REPLAY_MEMORY,len(E1_y)),np.random.default_rng(SEED)); ref=E1_X.copy(); prev=current(E1_X); prev_alert=float(np.mean(prev>=static_thr)); rows=[]
    for name in list(EXPERIENCES)[1:]:
        d=stream[name]; scores=current(d["X_adapt"]); fd=drift_summary(ref,d["X_adapt"]); sd=psi1(prev,scores); u=uncertainty(scores); inst=instability(current,d["X_adapt"]); alert=float(np.mean(scores>=static_thr)); ashift=abs(alert-prev_alert); requested=need_adaptation(fd,sd,u,inst,ashift); action=resource_action(requested,profile); accepted=False; rollback=False; ctime=0.
        if action in ["LIGHT_SPECIALIST","REPLAY_SPECIALIST"]:
            mb=min(LIGHT_MEMORY if action=="LIGHT_SPECIALIST" else REPLAY_MEMORY,profile["memory_budget"]); est=80 if action=="LIGHT_SPECIALIST" else 180; mx,my=update_memory(memX,memy,d["X_adapt"],d["y_adapt"],mb); sp,ctime,_=fit_specialist(d["X_adapt"],d["y_adapt"],mx,my,est); a=choose_alpha(fd,u,inst); cand=make_candidate_predictor(current,sp,a); g=gate(current,cand); accepted=bool(g["accepted"]); rollback=not accepted
            if accepted: current=cand; memX,memy=update_memory(memX,memy,d["X_adapt"],d["y_adapt"],REPLAY_MEMORY); ref=memX.copy()
        rows.append({"scenario":profile_name,"experience":name,"requested":requested,"resource_action":action,"accepted":accepted,"rollback":rollback,"p90_feature_psi":fd["p90_psi"],"score_psi":sd,"high_uncertainty_fraction":u["high_uncertainty_fraction"],"instability":inst,"alert_rate_shift":ashift,"candidate_time_sec":ctime,"time_budget_sec":profile["time_budget_sec"],"memory_budget":profile["memory_budget"]})
        prev=current(d["X_adapt"]); prev_alert=float(np.mean(prev>=static_thr))
    return pd.DataFrame(rows)

resource_scenarios=pd.concat([run_resource_profile(p) for p in RESOURCE_PROFILES],ignore_index=True); display(resource_scenarios.round(4)); resource_scenarios.to_csv(RESULTS/"phase2c_resource_scenarios.csv",index=False)


,scenario,experience,requested,resource_action,accepted,rollback,p90_feature_psi,score_psi,high_uncertainty_fraction,instability,alert_rate_shift,candidate_time_sec,time_budget_sec,memory_budget
0,generous,E2_high_attack,STRONG_ADAPT,REPLAY_SPECIALIST,True,False,2.2552,6.2078,0.1498,0.2283,0.4897,1.9211,60.0,8000
1,generous,E3_transition,STRONG_ADAPT,REPLAY_SPECIALIST,True,False,0.4442,0.1371,0.1446,0.1390,0.0848,0.8780,60.0,8000
2,generous,E4_generic_dominant,STRONG_ADAPT,REPLAY_SPECIALIST,True,False,1.6100,1.5002,0.3608,0.2137,0.0921,2.2981,60.0,8000
3,generous,E5_late,STRONG_ADAPT,REPLAY_SPECIALIST,True,False,1.1933,0.0465,0.3294,0.0940,0.0095,1.5842,60.0,8000
4,moderate,E2_high_attack,STRONG_ADAPT,LIGHT_SPECIALIST,True,False,2.2552,6.2078,0.1498,0.2283,0.4897,0.8538,20.0,4000
5,moderate,E3_transition,STRONG_ADAPT,LIGHT_SPECIALIST,True,False,0.4442,0.1395,0.1498,0.1373,0.0872,0.3166,20.0,4000
6,moderate,E4_generic_dominant,STRONG_ADAPT,LIGHT_SPECIALIST,True,False,1.6100,1.4961,0.3725,0.2180,0.0729,0.2628,20.0,4000
7,moderate,E5_late,STRONG_ADAPT,LIGHT_SPECIALIST,False,True,1.1933,0.0759,0.3362,0.1180,0.0073,0.4033,20.0,4000
8,constrained,E2_high_attack,STRONG_ADAPT,LIGHT_SPECIALIST,True,False,2.2552,6.2078,0.1498,0.2283,0.4897,3.1119,10.0,1000
9,constrained,E3_transition,STRONG_ADAPT,LIGHT_SPECIALIST,True,False,0.4442,0.1395,0.1498,0.1373,0.0872,0.3194,10.0,1000


# Final independent temporal test

The final test split is now evaluated once. No final-test label is used by any preceding decision.


In [19]:
# Final test comparison
static_test=metrics(YTEST,static_model.predict_proba(XTEST)[:,1],static_thr)
blind_test=metrics(YTEST,blind.predict_proba(XTEST)[:,1],blind_thr)
replay_test=metrics(YTEST,replay.predict_proba(XTEST)[:,1],replay_thr)
ewc_test=metrics(YTEST,mlp_probs(ewc,XTEST),ewc_thr)
final_predictor=adaptive_predictors["E5_late"]; adaptive_test=metrics(YTEST,final_predictor(XTEST),static_thr)
final_test=pd.DataFrame([{**{"method":"Static_LightGBM"},**static_test},{**{"method":"Blind_CL_LightGBM"},**blind_test},{**{"method":"Replay_LightGBM"},**replay_test},{**{"method":"EWC_MLP"},**ewc_test},{**{"method":"Proposed_Adaptive_IDS"},**adaptive_test}])
display(final_test[["method","accuracy","precision","recall","f1","fpr","specificity","balanced_accuracy","roc_auc","pr_auc"]].round(6)); final_test.to_csv(RESULTS/"final_temporal_test_comparison.csv",index=False)


,method,accuracy,precision,recall,f1,fpr,specificity,balanced_accuracy,roc_auc,pr_auc
0,Static_LightGBM,0.797357,0.903948,0.707094,0.793494,0.092054,0.907946,0.807520,0.921488,0.931174
1,Blind_CL_LightGBM,0.449400,0.000000,0.000000,0.000000,0.000000,1.000000,0.500000,0.500000,0.550600
2,Replay_LightGBM,0.867330,0.817678,0.976860,0.890209,0.266865,0.733135,0.854997,0.977204,0.982956
3,EWC_MLP,0.561726,0.556795,1.000000,0.715309,0.975243,0.024757,0.512378,0.711078,0.680494
4,Proposed_Adaptive_IDS,0.886885,0.843817,0.975029,0.904690,0.221108,0.778892,0.876960,0.948668,0.956139


In [20]:
# Final research summary + protocol
summary=pd.DataFrame([{
    "dataset":DATASET_ID,"configuration":DATASET_CONFIG,"main_resource_profile":MAIN_RESOURCE_PROFILE,"adaptive_f1":adaptive_test["f1"],"adaptive_recall":adaptive_test["recall"],"adaptive_precision":adaptive_test["precision"],"adaptive_fpr":adaptive_test["fpr"],"adaptive_balanced_accuracy":adaptive_test["balanced_accuracy"],"max_forgetting":float(forgetting.forgetting.max()) if len(forgetting) else np.nan,"accepted_adaptations":int(adaptive_decisions.accepted.sum()),"rollbacks":int(adaptive_decisions.rollback.sum()),"candidate_compute_sec":float(adaptive_decisions.candidate_time_sec.sum())
}]); display(summary.round(6)); summary.to_csv(RESULTS/"final_research_summary.csv",index=False)

protocol={"project":"CARC-IDS","dataset":DATASET_ID,"config":DATASET_CONFIG,"seed":SEED,"experiences":EXPERIENCES,"e1_split":{"train":.56,"calibration":.07,"protected":.07,"evaluation":.30},"later_split":{"adaptation":.70,"evaluation":.30,"chronological":True},"preprocessing_fit":"E1 training only","threshold_source":"method-specific E1 calibration; frozen thereafter","proposed_signals":["feature_PSI","score_PSI","entropy_uncertainty","prediction_instability","alert_rate_shift"],"proposed_actions":["NO_UPDATE","LIGHT_SPECIALIST","REPLAY_SPECIALIST"],"alpha_range":[ALPHA_MIN,ALPHA_MAX],"retention":{"max_f1_drop":MAX_F1_DROP,"max_recall_drop":MAX_RECALL_DROP,"max_protected_fpr":MAX_PROTECTED_FPR},"resource_profiles":RESOURCE_PROFILES,"baselines":["Static_LightGBM","Blind_CL_LightGBM","Replay_LightGBM","EWC_MLP"],"final_test_used_for_training":False,"final_test_used_for_threshold_tuning":False,"ips_implemented":False,"local_llm_implemented":False}
with open(RESULTS/"final_protocol.json","w") as f: json.dump(protocol,f,indent=2)
for name,obj in [("static_lightgbm",static_model),("blind_continual_lightgbm",blind),("replay_lightgbm",replay)]: joblib.dump(obj,ARTIFACTS/(name+".joblib"))
torch.save(ewc.state_dict(),ARTIFACTS/"ewc_mlp_state.pt")
manifest=pd.DataFrame([{"file":p.name,"bytes":p.stat().st_size} for p in sorted(RESULTS.glob("*"))]); manifest.to_csv(RESULTS/"output_manifest.csv",index=False)
print("\nCanonical outputs:"); [print(p) for p in sorted(RESULTS.glob("*"))]


,dataset,configuration,main_resource_profile,adaptive_f1,adaptive_recall,adaptive_precision,adaptive_fpr,adaptive_balanced_accuracy,max_forgetting,accepted_adaptations,rollbacks,candidate_compute_sec
0,lacg030175/UNSW-NB15,temporal,moderate,0.90469,0.975029,0.843817,0.221108,0.87696,0.082757,3,1,3.308756



Canonical outputs:
/content/CARC_IDS_FINAL/results/final_protocol.json
/content/CARC_IDS_FINAL/results/final_research_summary.csv
/content/CARC_IDS_FINAL/results/final_temporal_test_comparison.csv
/content/CARC_IDS_FINAL/results/output_manifest.csv
/content/CARC_IDS_FINAL/results/phase1_experience_summary.csv
/content/CARC_IDS_FINAL/results/phase1_segment_summary.csv
/content/CARC_IDS_FINAL/results/phase2a_attack_category_js.csv
/content/CARC_IDS_FINAL/results/phase2a_feature_drift_summary.csv
/content/CARC_IDS_FINAL/results/phase2a_numeric_feature_drift.csv
/content/CARC_IDS_FINAL/results/phase2c_adaptive_decisions.csv
/content/CARC_IDS_FINAL/results/phase2c_adaptive_stream_results.csv
/content/CARC_IDS_FINAL/results/phase2c_efficiency.csv
/content/CARC_IDS_FINAL/results/phase2c_forgetting.csv
/content/CARC_IDS_FINAL/results/phase2c_resource_scenarios.csv
/content/CARC_IDS_FINAL/results/static_threshold_selection.csv


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

# Interpretation / go-no-go

The canonical result is **not** judged by F1 alone. Compare: F1, recall, precision, FPR, balanced accuracy, forgetting, accepted updates, rollbacks, compute cost, and resource sensitivity.

If the proposed controller improves detection while keeping FPR and forgetting within acceptable bounds, the IDS methodology can be frozen and the next project stage can add the IPS response layer. If it fails, fix the IDS rather than tuning until a favorable number appears.

The current notebook does not claim production readiness, multi-dataset generalization, IPS safety, or LLM explanation quality.
